# LangGraph with AgentCore Memory Tool (Short term memory)

## Introduction
This notebook demonstrates how to integrate Amazon Bedrock AgentCore Memory capabilities with a conversational AI agent using LangGraph framework. We'll focus on **short-term memory** retention within a single conversation session - allowing an agent to recall information from earlier in the conversation without explicit context management.


## Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Personal Fitness                                                                 |
| Agentic Framework   | Langgraph                                                                        |
| LLM model           | Anthropic Claude Sonnet 3.7                                                      |
| Tutorial components | AgentCore Short-term Memory, Langgraph, Memory retrieval via Tool                |
| Example complexity  | Beginner                                                                         |

You'll learn to:
- Create a memory store with AgentCore Memory for short-term memory
- Use LangGraph to create an agent with structured memory workflows
- Implement memory tools for conversation history retrieval
- Access and utilize contextual information within a single session
- Enhance conversational experiences through effective memory recall


### Scenario Context

In this example, we'll create a "**Personal Fitness Coach**" that can remember workout details, fitness goals, physical limitations, and exercise preferences as they are mentioned throughout the conversation. This assistant will demonstrate how effective short-term memory management enables a more natural and personalized fitness coaching experience without requiring users to repeatedly state their information.


## Architecture
<div style="text-align:left">
    <img src="images/architecture.png" width="65%" />
</div>

## Prerequisites

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

Let's get started by setting up our environment!

## Step 1: Environment Setup
Let's begin importing all the necessary libraries and defining the clients to make this notebook work.

In [1]:
!pip install -qr requirements.txt

In [2]:
import logging
from datetime import datetime

Define the region and the role with the appropiate permissions for Amazon Bedrock models and AgentCore

In [3]:
import os
REGION = os.getenv('AWS_REGION', 'us-west-2')

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

### How the Integration Works

The integration between LangGraph and AgentCore Memory involves:

1. Using AgentCore Memory to store conversations in the short term memory
2. Structured workflows in LangGraph to manage memory operations

This approach separates memory management from reasoning, creating a cleaner and more maintainable agent architecture.

## Step 2: Memory Creation
In this section, we'll create a memory store using the AgentCore Starter Toolkit Memory SDK. This memory store will allow our agent to retain information from the conversation.

In [4]:
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from botocore.exceptions import ClientError

from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

In [5]:
memory_manager = MemoryManager(region_name=REGION)
memory_name = "FitnessCoach"
memory_id = None

✅ MemoryManager initialized for region: us-west-2
2025-10-20 23:35:32 - INFO - ✅ MemoryManager initialized for region: us-west-2


In [6]:
try:
    print("Creating Memory...")
    # Create the memory resource
    memory = memory_manager.get_or_create_memory(
        name=memory_name,                       # This name is unique across all memories in this account
        description="Fitness Coach Agent",      # Human-readable description
        strategies=[],                          # No memory strategies for short-term memory
        event_expiry_days=7,                    # Memories expire after 7 days
    )

    # Extract and print the memory ID
    memory_id = memory['id']
    logger.info(f"Memory created successfully with ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            memory_manager.delete_memory(memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

Creating Memory...


Memory already exists. Using existing memory ID: FitnessCoach-aBeeW578TQ
2025-10-20 23:35:38 - INFO - Memory already exists. Using existing memory ID: FitnessCoach-aBeeW578TQ
🔎 Retrieving memory resource with ID: FitnessCoach-aBeeW578TQ...
2025-10-20 23:35:38 - INFO - 🔎 Retrieving memory resource with ID: FitnessCoach-aBeeW578TQ...
  Found memory: FitnessCoach-aBeeW578TQ
2025-10-20 23:35:38 - INFO -   Found memory: FitnessCoach-aBeeW578TQ
Universal strategy validation passed for memory FitnessCoach. Strategies match: []
2025-10-20 23:35:38 - INFO - Universal strategy validation passed for memory FitnessCoach. Strategies match: []
2025-10-20 23:35:38 - INFO - Memory created successfully with ID: FitnessCoach-aBeeW578TQ


## Step 3: LangGraph Agent Creation
Let's import all the libraries we need to create the agent with LangGraph.

In [7]:
from langgraph.graph import StateGraph, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_aws import ChatBedrock

### LangGraph Agent Implementation

Now let's create the agent with LangGraph, incorporating our memory tools:

In [8]:
def create_agent(memory_session):
    """Create and configure the LangGraph agent"""
    
    # Initialize your LLM (adjust model and parameters as needed)
    llm = ChatBedrock(
        model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # or your preferred model
        model_kwargs={"temperature": 0.1}
    )
    
    @tool
    def list_events():
        """Tool used when needed to retrieve recent information""" 
        recent_turns = memory_session.get_last_k_turns(k=5)

        context = ""
        if recent_turns:
            # Format conversation history for context
            context_messages = []
            for turn in recent_turns:
                for message in turn:
                    # Handle both EventMessage objects and dict formats
                    if hasattr(message, 'role') and hasattr(message, 'content'):
                        role = message['role']
                        content = message['content']
                    else:
                        role = message.get('role', 'unknown')
                        content = message.get('content', {}).get('text', '')
                    context_messages.append(f"{role}: {content}")
            
            context = "\n".join(context_messages)
            logger.info(f"Context from memory: {context}")
        return context
        
    
    # Bind tools to the LLM
    tools = [list_events]
    llm_with_tools = llm.bind_tools(tools)
    
    # System message
    system_message = """You are the Personal Fitness Coach, a sophisticated fitness guidance assistant.
                        PURPOSE:
                        - Help users develop workout routines based on their fitness goals
                        - Remember user's exercise preferences, limitations, and progress
                        - Provide personalized fitness recommendations and training plans
                        MEMORY CAPABILITIES:
                        - You have access to recent events with the list_events tool
                        """
    
    # Define the chatbot node
    def chatbot(state: MessagesState):
        raw_messages = state["messages"]
    
        # Remove any existing system messages to avoid duplicates or misplacement
        non_system_messages = [msg for msg in raw_messages if not isinstance(msg, SystemMessage)]
    
        # Always ensure SystemMessage is first
        messages = [SystemMessage(content=system_message)] + non_system_messages
    
        latest_user_message = next((msg.content for msg in reversed(messages) if isinstance(msg, HumanMessage)), None)
    
        # Get response from model with tools bound
        response = llm_with_tools.invoke(messages)
    
        # Save conversation if applicable
        if latest_user_message.strip() and response.content.strip():
            try:
                memory_session.add_turns(
                    messages=[
                        ConversationalMessage(latest_user_message, MessageRole.USER),
                        ConversationalMessage(response.content, MessageRole.ASSISTANT),
                    ]
                )
            except Exception as e:
                print(f"Error saving conversation: {str(e)}")
        
        # Append response to full message history
        return {"messages": raw_messages + [response]}
    
    # Create the graph
    graph_builder = StateGraph(MessagesState)
    
    # Add nodes
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition,
    )
    graph_builder.add_edge("tools", "chatbot")
    
    # Set entry point
    graph_builder.set_entry_point("chatbot")
    
    # Compile the graph
    return graph_builder.compile()

### Creating a Wrapper for Agent Invocation

Let's create a simple wrapper to invoke our agent:

In [9]:
def langgraph_bedrock(payload, agent):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    
    # Create the input in the format expected by LangGraph
    response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
    
    # Extract the final message content
    return response["messages"][-1].content

## Step 4: Run the LangGraph Agent
We can now run the agent with our AgentCore Memory integration.

In [10]:
# Create unique actor and session IDs for this conversation
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"workout-{datetime.now().strftime('%Y%m%d%H%M%S')}"

In [11]:
# Create the agent with AgentCore Memory integration
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
memory_session = session_manager.create_memory_session(
            actor_id=actor_id, 
            session_id=session_id
        )
agent = create_agent(memory_session)

2025-10-20 23:36:00 - INFO - 💬 Creating new conversation for actor 'user-20251020233558' in session 'workout-20251020233558'...


#### Congratulations ! Your Agent is ready !!

### Let's test the Agent

Let's interact with our agent to test its memory capabilities:

In [12]:
response = langgraph_bedrock({"prompt": "Hello! This is my first day, I need a workout routine."}, agent)
print(f"Agent: {response}\n")

2025-10-20 23:36:03 - INFO - Using Bedrock Invoke API to generate response
2025-10-20 23:36:07 - INFO -   -> Storing 2 messages in short-term memory...
2025-10-20 23:36:08 - INFO -      ✅ Turn stored successfully with Event ID: None


Agent: Hi there! Welcome to your first day of fitness coaching. I'd be happy to help you develop a workout routine that's right for you. To create a personalized plan, I'll need to know a bit more about:

1. Your fitness goals (weight loss, muscle gain, general fitness, etc.)
2. Your current fitness level (beginner, intermediate, advanced)
3. Any equipment you have access to (home equipment, gym membership, etc.)
4. Any physical limitations or injuries I should be aware of
5. How many days per week you can commit to working out
6. Your preferred workout duration

Once you share these details, I can create a tailored workout routine that will help you achieve your specific fitness goals. Would you mind providing this information?



In [13]:
response = langgraph_bedrock({"prompt": "I want to build muscle, looking for a biceps routine. I have some lower back problems."}, agent)
print(f"Agent: {response}\n")

2025-10-20 23:36:20 - INFO - Using Bedrock Invoke API to generate response
2025-10-20 23:36:22 - INFO -   -> Storing 2 messages in short-term memory...
2025-10-20 23:36:23 - INFO -      ✅ Turn stored successfully with Event ID: None
2025-10-20 23:36:23 - INFO - Retrieved total of 2 events
2025-10-20 23:36:23 - INFO - Context from memory: USER: {'text': 'I want to build muscle, looking for a biceps routine. I have some lower back problems.'}
ASSISTANT: {'text': "I'd be happy to help you create a biceps routine that builds muscle while being mindful of your lower back problems. Let me check if I have any previous information about your fitness journey that might be relevant."}
USER: {'text': 'Hello! This is my first day, I need a workout routine.'}
ASSISTANT: {'text': "Hi there! Welcome to your first day of fitness coaching. I'd be happy to help you develop a workout routine that's right for you. To create a personalized plan, I'll need to know a bit more about:\n\n1. Your fitness goals 

Agent: Thank you for providing more details about your fitness goals. I see this is one of our first conversations, and you're specifically looking for a biceps routine to build muscle while accommodating lower back problems.

Here's a specialized biceps routine that will help you build muscle while minimizing stress on your lower back:

## Biceps Routine for Muscle Building (Lower Back Friendly)

**Frequency:** 2-3 times per week (allow 48 hours between biceps workouts)

**Warm-up:**
- 5 minutes of light cardio (walking or stationary bike at low resistance)
- Arm circles and dynamic stretches for upper body (30 seconds each)

**Main Workout:**

1. **Seated Dumbbell Curls** (seated position supports your back)
   - 3 sets of 10-12 reps
   - Use a bench with back support
   - Focus on controlled movement

2. **Cable Curls** (stable standing position)
   - 3 sets of 12-15 reps
   - Keep core engaged but not strained
   - Maintain neutral spine position

3. **Concentration Curls** (seated

In [14]:
response = langgraph_bedrock({"prompt": "Can you give me three exercises with number of reps?"}, agent)
print(f"Agent: {response}\n")

2025-10-20 23:36:41 - INFO - Using Bedrock Invoke API to generate response
2025-10-20 23:36:44 - INFO -   -> Storing 2 messages in short-term memory...
2025-10-20 23:36:44 - INFO -      ✅ Turn stored successfully with Event ID: None
2025-10-20 23:36:44 - INFO - Retrieved total of 4 events
2025-10-20 23:36:44 - INFO - Context from memory: USER: {'text': 'Can you give me three exercises with number of reps?'}
ASSISTANT: {'text': "I'd be happy to suggest three exercises with recommended repetitions for you. To make these recommendations more personalized, I should understand a bit more about your fitness goals, experience level, and any equipment you have available."}
USER: {'text': 'I want to build muscle, looking for a biceps routine. I have some lower back problems.'}
ASSISTANT: {'text': "Thank you for providing more details about your fitness goals. I see this is one of our first conversations, and you're specifically looking for a biceps routine to build muscle while accommodating lo

Agent: Based on our previous conversations, I understand you're looking for a biceps routine to build muscle while being mindful of your lower back problems. Here are three effective biceps exercises with recommended repetitions:

1. **Seated Dumbbell Curls**
   - 3 sets of 10-12 repetitions
   - Perform while seated with back support to protect your lower back
   - Focus on controlled movement with full range of motion

2. **Cable Curls**
   - 3 sets of 12-15 repetitions
   - Maintain a stable standing position with neutral spine
   - Keep core engaged but not strained to support your lower back

3. **Concentration Curls**
   - 3 sets of 10-12 repetitions per arm
   - Perform seated on a bench with elbow braced against inner thigh
   - This isolated movement minimizes stress on your lower back

For all exercises, rest 60-90 seconds between sets. Start with lighter weights to perfect your form before increasing the load. These exercises are specifically chosen to help you build bicep m

### Testing Memory Persistence

To truly demonstrate the power of the AgentCore Memory integration, let's create a new agent instance and see if it can recall our previous conversation:

In [15]:
# Create a new agent instance (simulating a new session)
new_agent = create_agent(memory_session)

# Test if the new agent remembers our preferences
response = langgraph_bedrock({
    "prompt": "Hello again! Can you remind me about my last workout session?"
}, new_agent)

print("New Agent Session:\n")
print(f"Agent: {response}")

2025-10-20 23:36:54 - INFO - Using Bedrock Invoke API to generate response
2025-10-20 23:36:57 - INFO -   -> Storing 2 messages in short-term memory...
2025-10-20 23:36:57 - INFO -      ✅ Turn stored successfully with Event ID: None
2025-10-20 23:36:57 - INFO - Retrieved total of 6 events
2025-10-20 23:36:57 - INFO - Context from memory: USER: {'text': 'Hello again! Can you remind me about my last workout session?'}
ASSISTANT: {'text': "I'd be happy to remind you about your last workout session. Let me check your recent fitness activities."}
USER: {'text': 'Can you give me three exercises with number of reps?'}
ASSISTANT: {'text': "Based on our previous conversations, I understand you're looking for a biceps routine to build muscle while being mindful of your lower back problems. Here are three effective biceps exercises with recommended repetitions:\n\n1. **Seated Dumbbell Curls**\n   - 3 sets of 10-12 repetitions\n   - Perform while seated with back support to protect your lower back

New Agent Session:

Agent: Based on your recent interactions, I can see that we've discussed a biceps routine tailored to your specific needs. Your last workout-related conversation focused on developing a biceps routine to build muscle while accommodating your lower back problems.

I recommended the following exercises for you:

1. **Seated Dumbbell Curls**
   - 3 sets of 10-12 repetitions
   - Performed seated with back support to protect your lower back

2. **Cable Curls**
   - 3 sets of 12-15 repetitions
   - With a stable standing position and neutral spine

3. **Concentration Curls**
   - 3 sets of 10-12 repetitions per arm
   - Performed seated with elbow braced against inner thigh

I also suggested a more comprehensive biceps routine that included these exercises plus Preacher Curls and a 21s finisher, with specific recommendations for rest periods (60-90 seconds between sets) and form guidance to protect your lower back.

Would you like to continue with this routine, make adju

## Summary

In this notebook, we've demonstrated:

1. How to create a AgentCore Memory resource for an AI agent
2. Building a LangGraph workflow with memory integration
3. Implementing memory tools for conversation history retrieval
4. Creating an agent that intelligently uses memory when needed
5. Testing memory persistence across agent instances

This integration showcases the power of combining structured workflows (LangGraph) with robust memory systems (AgentCore Memory) to create more intelligent and context-aware AI agents.

The approach we've demonstrated can be extended to more complex use cases, including multi-agent systems, long-term memory with extraction strategies, and specialized memory retrieval based on conversation context.

## Clean up
Let's delete the memory to clean up the resources used in this notebook.

In [ ]:
# Uncomment to delete memory resource using MemoryManager
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"✅ Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")